# Hearing Reality - Colab Cloud Runner

This notebook pulls the latest code from GitHub, mounts Google Drive to access the 30GB+ datasets, and runs the PyTorch training loop.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Remigaraki/TwoStream-Audio-Forensics.git
%cd TwoStream-Audio-Forensics
!conda env create -f environment.yml
!conda activate hearing_reality

In [ ]:
import torch
from src.fusion.two_stream_net import TwoStreamFusionNet

print('GPU Available:', torch.cuda.is_available())
model = TwoStreamFusionNet().cuda() if torch.cuda.is_available() else TwoStreamFusionNet()
print('Model Loaded successfully!')

In [ ]:
from torch.utils.data import DataLoader
from data.dataset_loader import ASVspoofDataset, fit_stream2_pipeline_from_protocol

# Replace these with the Google Drive paths used in your Colab session.
base_dir = '/content/drive/MyDrive/hearing_reality/train_audio'
protocol_file = '/content/drive/MyDrive/hearing_reality/train_protocol.txt'

stream2_pipeline = fit_stream2_pipeline_from_protocol(base_dir, protocol_file, max_items=256)
train_dataset = ASVspoofDataset(
    base_dir=base_dir,
    protocol_file=protocol_file,
    stream2_pipeline=stream2_pipeline,
    return_stream2=True,
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

audio_batch, stats_batch, labels_batch = next(iter(train_loader))
print(audio_batch.shape, stats_batch.shape, labels_batch.shape)
print(model(audio_batch, stats_batch).shape)

In [ ]:
from pathlib import Path

from src.training.train_loop import (
    TrainingConfig,
    build_two_stream_loaders,
    fit_two_stream_model,
    set_seed,
)
from src.utils.logger import TensorBoardLogger, setup_logger

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

train_loader, val_loader = build_two_stream_loaders(
    train_dataset,
    batch_size=8,
    val_fraction=0.2,
    seed=42,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

checkpoint_path = '/content/drive/MyDrive/hearing_reality/checkpoints/best_two_stream.pt'
log_dir = '/content/drive/MyDrive/hearing_reality/runs/colab_main'
Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
Path(log_dir).mkdir(parents=True, exist_ok=True)

logger = setup_logger('hearing_reality_train', str(Path(log_dir) / 'train.log'))
tb_logger = TensorBoardLogger(log_dir)

config = TrainingConfig(
    epochs=1,
    batch_size=8,
    learning_rate=1e-4,
    weight_decay=1e-4,
    val_fraction=0.2,
    seed=42,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    checkpoint_path=checkpoint_path,
    log_dir=log_dir,
)

history = fit_two_stream_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    logger=logger,
    tb_logger=tb_logger,
)

tb_logger.close()
print(history[-1])

In [ ]:
from torch.utils.data import DataLoader
from data.dataset_loader import ASVspoofDataset
from src.training.train_loop import evaluate_epoch

# Replace these with the Google Drive paths for your held-out evaluation split.
eval_base_dir = '/content/drive/MyDrive/hearing_reality/eval_audio'
eval_protocol_file = '/content/drive/MyDrive/hearing_reality/eval_protocol.txt'

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

eval_dataset = ASVspoofDataset(
    base_dir=eval_base_dir,
    protocol_file=eval_protocol_file,
    stream2_pipeline=stream2_pipeline,
    return_stream2=True,
    cache_stream2_features=True,
)
eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

eval_metrics = evaluate_epoch(
    model=model,
    loader=eval_loader,
    criterion=torch.nn.CrossEntropyLoss(),
    device=device,
)

print({
    'held_out_loss': eval_metrics['loss'],
    'held_out_eer': eval_metrics['eer'],
    'held_out_t_dcf': eval_metrics['t_dcf'],
    'held_out_eer_threshold': eval_metrics['eer_threshold'],
    'held_out_t_dcf_threshold': eval_metrics['t_dcf_threshold'],
})

### Start Training Loop Here

(Training logic utilizing data loaders from `/content/drive`...)
